# Lung Sounds, End to End
### IEEE BSN Workshop · hands-on session

You will take a real respiratory recording from **filter → segment → spectrogram → classifier → ONNX**,
changing every setting with menus and sliders rather than code.

**The dataset.** ICBHI 2017 (Rocha et al., 2019) — 920 chest recordings annotated cycle by cycle.
Each respiratory cycle carries two independent flags, *crackle* and *wheeze*, giving four classes:

| | Crackle | Wheeze |
|---|---|---|
| **Normal** | no | no |
| **Wheeze** | no | yes |
| **Crackle** | yes | no |
| **Both** | yes | yes |

**No GPU, no training from scratch.** The six "models" in section 6 are *pre-computed embeddings*:
each backbone was run over the data once, in advance, and its output vectors saved. Picking a model
from the dropdown just loads a different `.npy` file. All you train is a small classifier head on top —
20 epochs, a second or two on a CPU.

**Run order matters.** Run section 1 first, then go in order. Each section leaves something behind that
the next one uses.

---
## 1 | Setup

**What this does.** Installs anything missing, locates the data bundle, and loads `manifest.json` —
the file that records what the data *is*: the class order, each model's embedding size, the exact
preprocessing settings, and the known caveats.

**Why read the manifest.** The class order for this dataset is **not** the obvious one, so nothing in
this notebook hard-codes it. Everything downstream reads `LABELS` from the manifest.

In [ ]:
# --- environment -------------------------------------------------------------
import importlib, subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules

def ensure(pkg, import_name=None):
    """Install pkg only if it isn't importable. Keeps Colab startup quiet."""
    try:
        importlib.import_module(import_name or pkg)
        return True
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
        try:
            importlib.import_module(import_name or pkg)
            return True
        except ImportError:
            print(f"  could not import {import_name or pkg} after install")
            return False

for pkg, mod in [("numpy", "numpy"), ("scipy", "scipy"), ("librosa", "librosa"),
                 ("soundfile", "soundfile"), ("matplotlib", "matplotlib"),
                 ("ipywidgets", "ipywidgets")]:
    ensure(pkg, mod)

if IN_COLAB:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()

import json, glob, math, warnings
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, Audio, clear_output

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "figure.autolayout": True,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 9})
print(f"python {sys.version.split()[0]} | numpy {np.__version__} | colab={IN_COLAB}")

In [ ]:
# --- locate the data ---------------------------------------------------------
# Priority: BSN_DATA_DIR env var  ->  ./bsn_workshop_data  ->  download from Hugging Face.
HF_REPO, HF_FOLDER = "AlirezaFB/ICBHI_2017", "workshop"

HELP = f"""
Could not find the data bundle.

Fix it in one of two ways:

  1. Point at a local copy:
         import os; os.environ["BSN_DATA_DIR"] = r"/path/to/bsn_workshop_data"
     then re-run this cell. The folder must contain manifest.json.

  2. Make sure {HF_REPO} (folder {HF_FOLDER}/) is reachable and public,
     or log in first with:  from huggingface_hub import login; login()
"""

def _looks_like_bundle(d):
    return bool(d) and os.path.isfile(os.path.join(d, "manifest.json"))

def find_data_dir():
    # 1. explicit override
    if _looks_like_bundle(os.environ.get("BSN_DATA_DIR")):
        return os.environ["BSN_DATA_DIR"]

    # 2. look beside the notebook and a couple of levels up
    here = os.path.abspath(os.getcwd())
    for up in range(3):
        base = here if up == 0 else os.path.abspath(os.path.join(here, *([".."] * up)))
        for sub in ("bsn_workshop_data", os.path.join("data", "bsn_workshop_data")):
            c = os.path.join(base, sub)
            if _looks_like_bundle(c):
                return c
    if _looks_like_bundle("/content/bsn_workshop_data"):
        return "/content/bsn_workshop_data"

    # 3. Hugging Face
    if not ensure("huggingface_hub"):
        raise RuntimeError("huggingface_hub is unavailable." + HELP)
    from huggingface_hub import snapshot_download
    print(f"no local bundle found; downloading {HF_REPO}:{HF_FOLDER}/ ...")
    try:
        root = snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                                 allow_patterns=f"{HF_FOLDER}/*")
    except Exception as e:
        raise RuntimeError(f"Hugging Face download failed ({type(e).__name__}: "
                           f"{str(e).splitlines()[0]})." + HELP) from None
    d = os.path.join(root, HF_FOLDER)
    if not _looks_like_bundle(d):
        raise RuntimeError(f"Downloaded {HF_REPO} but {HF_FOLDER}/manifest.json is missing." + HELP)
    return d

DATA_DIR = find_data_dir()
MANIFEST = json.load(open(os.path.join(DATA_DIR, "manifest.json"), encoding="utf-8"))

# Class order comes from the manifest. Never hard-code it.
LABELS   = [MANIFEST["label_map"][str(i)] for i in range(len(MANIFEST["label_map"]))]
LABEL_IX = {name: i for i, name in enumerate(LABELS)}
NORMAL   = LABEL_IX["Normal"]
ABNORMAL = [i for i, n in enumerate(LABELS) if n != "Normal"]

# Defaults for the DSP sections, also from the manifest.
_pp  = MANIFEST["preprocessing"]
SR_D = int(_pp["resample_rate_hz"])
LO_D, HI_D = float(_pp["filter"]["lowcut_hz"]), float(_pp["filter"]["highcut_hz"])
ORD_D      = int(_pp["filter"]["order"])

print(f"data: {DATA_DIR}")
print(f"class order (from manifest): {dict(enumerate(LABELS))}")
print(f"default filter: Butterworth band-pass {LO_D:.0f}-{HI_D:.0f} Hz, order {ORD_D}, at {SR_D} Hz")
print(f"\nmodels available: {', '.join(MANIFEST['models'])}")

In [ ]:
# --- the 12 sample recordings ------------------------------------------------
RECS = {r["recording"]: r for r in MANIFEST["sample_recordings"]["recordings"]}
DEVICES = sorted({r["device"] for r in RECS.values()})

print(f"{len(RECS)} recordings from {len(DEVICES)} recording devices\n")
print(f"{'recording':26s} {'device':10s} {'split':6s} {'SR':>6s} {'sec':>6s}   " +
      " ".join(f"{n[:4]:>5s}" for n in LABELS))
for name, r in RECS.items():
    c = r["cycles_per_class"]
    print(f"{name:26s} {r['device']:10s} {r['split']:6s} "
          f"{r['original_sample_rate_hz']:6d} {r['duration_s']:6.1f}   " +
          " ".join(f"{c[n]:5d}" for n in LABELS))

print("\nNote the sample rates differ (44100 Hz and 4000 Hz). Code that assumes one rate breaks here,")
print("so every helper below reads the rate from the file.")

---
## 2 | Listen to the data

**What this does.** Draws one recording's waveform with every annotated cycle shaded by its class,
and gives you a player.

**What you can try.** Switch device — the four microphones sound genuinely different. Tick *hide normal
cycles* to see where the abnormal events actually sit. Crackles are short clicks; wheezes are longer,
more musical.

**Listen for the microphone, not just the lung.** `AKGC417L` and `Meditron` are bright; `Litt3200`
recordings arrive at 4 kHz already and sound muffled. A classifier can learn the microphone instead of
the pathology — that is the central difficulty of this dataset.

In [ ]:
import soundfile as sf
import librosa
from scipy.signal import butter, sosfiltfilt, sosfilt

PLAY_SR = 8000          # cap playback rate: keeps the notebook light
CLR = {"Normal": "#9aa5ad", "Wheeze": "#0b6e75", "Crackle": "#c2622a", "Both": "#8e3b8e"}

def load_recording(name):
    """Return (signal float64 mono, sample_rate). No resampling."""
    y, sr = sf.read(os.path.join(DATA_DIR, "audio", f"{name}.wav"), dtype="float64")
    if y.ndim > 1:
        y = y.mean(axis=1)
    return y, sr

def load_cycles(name):
    """Parse the ICBHI annotation: start, end, crackle, wheeze -> (start, end, class_int).

    The class integer is resolved through the manifest's label_map, so it stays correct
    even if the mapping were ever changed upstream.
    """
    out = []
    with open(os.path.join(DATA_DIR, "audio", f"{name}.txt")) as fh:
        for line in fh:
            p = line.split()
            if len(p) != 4:
                continue
            st, en, cr, wh = float(p[0]), float(p[1]), int(float(p[2])), int(float(p[3]))
            cr, wh = bool(cr), bool(wh)
            name_of = "Both" if (cr and wh) else "Crackle" if cr else "Wheeze" if wh else "Normal"
            out.append((st, en, LABEL_IX[name_of]))
    return out

def for_playback(y, sr, target=PLAY_SR):
    """Downsample for the audio widget only. Never used for analysis."""
    if sr <= target:
        return y, sr
    return librosa.resample(y, orig_sr=sr, target_sr=target), target

def show_recording(name, hide_normal=False):
    y, sr = load_recording(name)
    cyc   = load_cycles(name)
    t     = np.arange(len(y)) / sr
    fig, ax = plt.subplots(figsize=(11, 2.6))
    ax.plot(t, y, lw=0.4, color="#2b3036", zorder=3)
    seen = set()
    for st, en, k in cyc:
        nm = LABELS[k]
        if hide_normal and k == NORMAL:
            continue
        ax.axvspan(st, en, color=CLR[nm], alpha=0.30, lw=0, zorder=1,
                   label=nm if nm not in seen else None)
        seen.add(nm)
    ax.set_xlim(0, t[-1]); ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")
    ax.set_title(f"{name}   |   {RECS[name]['device']}   |   {sr} Hz   |   "
                 f"{len(cyc)} cycles" + ("   (normal cycles hidden)" if hide_normal else ""))
    if seen:
        ax.legend(loc="upper right", ncol=len(seen), fontsize=8, framealpha=0.9)
    plt.show()
    yp, sp = for_playback(y, sr)
    display(Audio(yp, rate=sp))

In [ ]:
# --- interactive ---
_w_dev  = W.Dropdown(options=DEVICES, value=DEVICES[0], description="Device:")
_w_rec  = W.Dropdown(description="Recording:", layout=W.Layout(width="420px"))
_w_hide = W.Checkbox(value=False, description="hide normal cycles", indent=False)
_out2   = W.Output()

def _recs_for(dev):
    return [n for n, r in RECS.items() if r["device"] == dev]

def _refresh2(*_):
    with _out2:
        clear_output(wait=True)
        if _w_rec.value:
            show_recording(_w_rec.value, _w_hide.value)

def _on_dev(*_):
    _w_rec.options = _recs_for(_w_dev.value)
    _w_rec.value = _w_rec.options[0]

_w_dev.observe(_on_dev, "value")
_w_rec.observe(_refresh2, "value")
_w_hide.observe(_refresh2, "value")
_on_dev()
display(W.VBox([W.HBox([_w_dev, _w_rec]), _w_hide, _out2]))
_refresh2()

### ✏️ Try it yourself

Pick the two recordings with the most *Both* cycles from the table in section 1 and listen back to back. Can you hear the difference between a crackle and a wheeze? Try hiding the normal cycles first so you know where to listen.

*(The cell below is yours — nothing here is graded.)*

In [ ]:
# Your turn. The helpers above are all available, e.g. load_recording, apply_filter, cut_cycle.

---
## 3 | Filtering

**What this does.** Resamples the recording, applies a Butterworth filter, and shows waveform and
spectrum before and after, with a player for each.

**What you can try.** Start from the defaults (band-pass 80–1900 Hz, order 5, 4 kHz) — the settings the
saved embeddings were made with. Then push the low cut up to 300 Hz and listen to the heart sounds
disappear. Drop the high cut to 500 Hz and the crackles go with it. Raise the order and watch the
transition sharpen.

**About the spectrum.** Plotted as amplitude in dB, `20·log10|X(f)|`, relative to the maximum of the
unfiltered signal so both panels share one scale. Stating the dB convention matters: amplitude
`20·log10` and power `10·log10` differ by a factor of two, and mixing them is the most common silent
bug in audio code.

**One honest caveat.** This section filters *zero-phase* (`sosfiltfilt`, forwards and backwards) because
that is what you want when inspecting a signal. The pipeline that produced the saved embeddings used a
single causal pass (`scipy.signal.lfilter`), which shifts phase. The magnitude spectra match; the
waveform shapes differ slightly.

In [ ]:
NYQ_MARGIN = 0.98   # keep cutoffs strictly below Nyquist

def apply_filter(y, sr, kind="band", lo=LO_D, hi=HI_D, order=ORD_D):
    """Butterworth filter, zero-phase. kind: none | low | high | band."""
    if kind == "none":
        return y.copy()
    nyq = 0.5 * sr
    lo_n, hi_n = lo / nyq, min(hi / nyq, NYQ_MARGIN)
    if kind == "low":
        if hi_n <= 0: return y.copy()
        sos = butter(order, hi_n, btype="low", output="sos")
    elif kind == "high":
        if lo_n <= 0: return y.copy()
        sos = butter(order, lo_n, btype="high", output="sos")
    else:
        if not (0 < lo_n < hi_n < 1):
            raise ValueError(f"band-pass needs 0 < {lo:.0f} < {hi:.0f} < {nyq:.0f} Hz")
        sos = butter(order, [lo_n, hi_n], btype="band", output="sos")
    # filtfilt needs a signal longer than the filter's padding
    if len(y) <= 3 * sos.shape[0] * 2:
        return sosfilt(sos, y)
    return sosfiltfilt(sos, y)

def spectrum_db(y, sr, ref=None):
    """Single-sided amplitude spectrum in dB: 20*log10|X|. Returns (freqs, dB, ref)."""
    n = int(2 ** math.ceil(math.log2(max(len(y), 2))))
    X = np.abs(np.fft.rfft(y * np.hanning(len(y)), n=n))
    f = np.fft.rfftfreq(n, 1.0 / sr)
    r = ref if ref is not None else max(X.max(), 1e-12)
    return f, 20 * np.log10(np.maximum(X, 1e-12) / r), r

def show_filter(name, kind, lo, hi, order, target_sr):
    y0, sr0 = load_recording(name)
    y = librosa.resample(y0, orig_sr=sr0, target_sr=target_sr) if sr0 != target_sr else y0.copy()
    sr = target_sr
    try:
        yf = apply_filter(y, sr, kind, lo, hi, order)
    except ValueError as e:
        print(f"invalid settings: {e}"); return
    t = np.arange(len(y)) / sr
    fig, ax = plt.subplots(2, 2, figsize=(11, 4.6))
    ax[0, 0].plot(t, y,  lw=0.4, color="#2b3036"); ax[0, 0].set_title(f"before  ({sr} Hz)")
    ax[0, 1].plot(t, yf, lw=0.4, color="#0b6e75"); ax[0, 1].set_title(f"after  ({kind})")
    for a in ax[0]:
        a.set_xlabel("Time (s)"); a.set_ylabel("Amplitude"); a.set_xlim(0, t[-1])
    lim = max(np.abs(y).max(), np.abs(yf).max()) * 1.05
    ax[0, 0].set_ylim(-lim, lim); ax[0, 1].set_ylim(-lim, lim)

    f1, d1, ref = spectrum_db(y, sr)
    f2, d2, _   = spectrum_db(yf, sr, ref=ref)
    for a, (f, d, c, ttl) in zip(ax[1], [(f1, d1, "#2b3036", "spectrum before"),
                                         (f2, d2, "#0b6e75", "spectrum after")]):
        a.plot(f, d, lw=0.7, color=c)
        a.set_xlim(0, sr / 2); a.set_ylim(-90, 2)
        a.set_xlabel("Frequency (Hz)"); a.set_ylabel("Amplitude (dB re max)")
        a.set_title(ttl)
        if kind in ("high", "band"):
            a.axvline(lo, color="#c2622a", ls="--", lw=0.9)
        if kind in ("low", "band"):
            a.axvline(min(hi, sr / 2 * NYQ_MARGIN), color="#c2622a", ls="--", lw=0.9)
    plt.show()
    yp, sp = for_playback(y, sr);  print("before:"); display(Audio(yp, rate=sp))
    yp, sp = for_playback(yf, sr); print("after:");  display(Audio(yp, rate=sp))

In [ ]:
# --- interactive ---
_f_rec  = W.Dropdown(options=list(RECS), value=list(RECS)[0], description="Recording:",
                     layout=W.Layout(width="420px"))
_f_kind = W.Dropdown(options=[("none", "none"), ("low-pass", "low"),
                              ("high-pass", "high"), ("band-pass", "band")],
                     value="band", description="Filter:")
_f_lo   = W.FloatSlider(value=LO_D, min=10,  max=900,  step=10, description="low cut (Hz):",
                        continuous_update=False, readout_format=".0f", style={"description_width":"110px"})
_f_hi   = W.FloatSlider(value=HI_D, min=200, max=3900, step=50, description="high cut (Hz):",
                        continuous_update=False, readout_format=".0f", style={"description_width":"110px"})
_f_ord  = W.IntSlider(value=ORD_D, min=1, max=10, step=1, description="order:",
                      continuous_update=False)
_f_sr   = W.Dropdown(options=[4000, 8000, 16000], value=SR_D, description="sample rate:",
                     style={"description_width":"110px"})
_out3   = W.Output()

def _refresh3(*_):
    with _out3:
        clear_output(wait=True)
        show_filter(_f_rec.value, _f_kind.value, _f_lo.value, _f_hi.value,
                    _f_ord.value, _f_sr.value)

for _w in (_f_rec, _f_kind, _f_lo, _f_hi, _f_ord, _f_sr):
    _w.observe(_refresh3, "value")
display(W.VBox([W.HBox([_f_rec, _f_kind]), W.HBox([_f_lo, _f_hi]),
                W.HBox([_f_ord, _f_sr]), _out3]))
_refresh3()

### ✏️ Try it yourself

Set the filter to **high-pass at 300 Hz**. What disappears, and why is that a problem for a lung-sound classifier? Then try **4 kHz vs 16 kHz** at the same band-pass settings — look at where the spectrum ends in each case, and think about what the 4 kHz recordings can never show you.

*(The cell below is yours — nothing here is graded.)*

In [ ]:
# Your turn. The helpers above are all available, e.g. load_recording, apply_filter, cut_cycle.

---
## 4 | Segmentation

**What this does.** Cuts one annotated cycle out of a recording and forces it to a fixed length — the
step that turns variable-length cycles into fixed-size model input.

**What you can try.** Pick a short cycle and a long duration to see heavy padding, then compare
*reflect*, *zeros* and *repeat*. The padded region is shaded, so you can see exactly how much of the
"cycle" is invented. Then set a duration shorter than the cycle and watch truncation throw audio away.

**Why this matters.** The saved embeddings used **reflect** padding to **4 s** (8 s for OPERA). Reflect
padding duplicates real signal, so a model can key on the repetition; zero padding introduces edges
that look like transients; repeat creates artificial periodicity. There is no free option — only a
choice you should make deliberately and write down.

In [ ]:
def cut_cycle(y, sr, start_s, end_s, duration_s, mode="reflect"):
    """Cut [start_s, end_s) and force it to duration_s.

    Returns (segment, pad_start_samples, pad_end_samples). Padding is appended at the end,
    matching the pipeline that produced the embeddings.
    """
    a, b = int(start_s * sr), int(end_s * sr)
    seg  = y[max(a, 0):min(b, len(y))]
    want = int(round(duration_s * sr))
    if len(seg) == 0:
        return np.zeros(want), 0, want
    if len(seg) >= want:
        return seg[:want], 0, 0
    need = want - len(seg)
    if mode == "zeros":
        out = np.concatenate([seg, np.zeros(need)])
    elif mode == "repeat":
        reps = int(np.ceil(want / len(seg)))
        out = np.tile(seg, reps)[:want]
    else:  # reflect
        if need >= len(seg):
            reps = int(np.ceil(want / len(seg)))
            out = np.tile(np.concatenate([seg, seg[::-1]]), reps)[:want]
        else:
            out = np.pad(seg, (0, need), mode="reflect")
    return out, 0, need

def show_segment(name, cycle_ix, duration_s, mode, target_sr):
    y0, sr0 = load_recording(name)
    y  = librosa.resample(y0, orig_sr=sr0, target_sr=target_sr) if sr0 != target_sr else y0.copy()
    sr = target_sr
    y  = apply_filter(y, sr, "band", LO_D, HI_D, ORD_D)
    cyc = load_cycles(name)
    st, en, k = cyc[cycle_ix]
    seg, _, pad = cut_cycle(y, sr, st, en, duration_s, mode)
    real = len(seg) - pad
    t = np.arange(len(seg)) / sr
    fig, ax = plt.subplots(figsize=(11, 2.4))
    ax.plot(t, seg, lw=0.5, color="#2b3036", zorder=3)
    ax.axvspan(0, real / sr, color=CLR[LABELS[k]], alpha=0.22, lw=0, label="real audio")
    if pad:
        ax.axvspan(real / sr, len(seg) / sr, color="#c2622a", alpha=0.22, lw=0,
                   label=f"{mode} padding ({pad/sr:.2f} s)")
    truncated = (en - st) > duration_s
    ax.set_xlim(0, t[-1]); ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")
    ax.set_title(f"cycle {cycle_ix} of {len(cyc)}  |  {LABELS[k]}  |  "
                 f"annotated {en-st:.2f} s -> {duration_s:.1f} s"
                 + ("   TRUNCATED" if truncated else ""))
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    plt.show()
    pct = 100.0 * pad / len(seg)
    print(f"{real/sr:.2f} s real + {pad/sr:.2f} s padding   ({pct:.0f}% of this segment is invented)")
    yp, sp = for_playback(seg, sr); display(Audio(yp, rate=sp))

In [ ]:
# --- interactive ---
_s_rec  = W.Dropdown(options=list(RECS), value=list(RECS)[0], description="Recording:",
                     layout=W.Layout(width="420px"))
_s_cyc  = W.IntSlider(value=0, min=0, max=1, step=1, description="cycle #:", continuous_update=False)
_s_dur  = W.FloatSlider(value=4.0, min=0.5, max=10.0, step=0.5, description="duration (s):",
                        continuous_update=False, style={"description_width":"110px"})
_s_mode = W.Dropdown(options=[("reflect", "reflect"), ("zeros", "zeros"), ("repeat", "repeat")],
                     value="reflect", description="padding:")
_s_sr   = W.Dropdown(options=[4000, 8000, 16000], value=SR_D, description="sample rate:",
                     style={"description_width":"110px"})
_out4   = W.Output()

def _refresh4(*_):
    with _out4:
        clear_output(wait=True)
        show_segment(_s_rec.value, _s_cyc.value, _s_dur.value, _s_mode.value, _s_sr.value)

def _on_srec(*_):
    _s_cyc.max = max(len(load_cycles(_s_rec.value)) - 1, 0)
    _s_cyc.value = 0
    _refresh4()

_s_rec.observe(_on_srec, "value")
for _w in (_s_cyc, _s_dur, _s_mode, _s_sr):
    _w.observe(_refresh4, "value")
_on_srec()
display(W.VBox([W.HBox([_s_rec, _s_cyc]), W.HBox([_s_dur, _s_mode]), _s_sr, _out4]))

### ✏️ Try it yourself

Find the shortest cycle you can (try `175_1b1_Al_sc_Litt3200`) and set the duration to 8 s. What fraction of the segment is now invented? Would you trust a model trained on that? Compare *reflect* and *zeros* at that setting.

*(The cell below is yours — nothing here is graded.)*

In [ ]:
# Your turn. The helpers above are all available, e.g. load_recording, apply_filter, cut_cycle.

---
## 5 | Mel spectrogram

**What this does.** Turns one segment into the grey image that the three vision backbones were fed.

**What you can try.** `n_fft` trades time resolution against frequency resolution — small windows localise
crackles, large windows resolve the harmonics of a wheeze. You cannot have both; that is the uncertainty
principle, not a shortcoming of the code. Drop `n_mels` to 32 and watch detail vanish. Pull `fmax` down
to 1000 Hz and the top of the image goes empty.

**About the dB here.** `librosa.power_to_db` is a *power* conversion, `10·log10`, not the `20·log10`
amplitude convention used in section 3. Same data, different convention — which is exactly why each
plot says which one it uses.

**Size note.** The images actually stored for this project are **251 × 128** (time × mel). Backbones
resize to their own expected input, 224 × 224 for the torchvision models. The square preview below is
the resized view.

In [ ]:
def mel_image(seg, sr, n_fft=1024, hop=256, n_mels=128, fmax=None):
    """Mel spectrogram in dB (power convention, 10*log10). Returns (image, extent)."""
    fmax = float(fmax or sr / 2)
    fmax = min(fmax, sr / 2)
    n_fft = min(int(n_fft), max(len(seg), 64))
    M = librosa.feature.melspectrogram(y=seg, sr=sr, n_fft=n_fft, hop_length=int(hop),
                                       n_mels=int(n_mels), fmin=0.0, fmax=fmax, power=2.0)
    D = librosa.power_to_db(M, ref=np.max)
    return D, [0, len(seg) / sr, 0, fmax]

def show_mel(name, cycle_ix, duration_s, n_fft, hop, n_mels, fmax, target_sr):
    y0, sr0 = load_recording(name)
    y  = librosa.resample(y0, orig_sr=sr0, target_sr=target_sr) if sr0 != target_sr else y0.copy()
    sr = target_sr
    y  = apply_filter(y, sr, "band", LO_D, HI_D, ORD_D)
    cyc = load_cycles(name)
    st, en, k = cyc[min(cycle_ix, len(cyc) - 1)]
    seg, _, pad = cut_cycle(y, sr, st, en, duration_s, "reflect")
    D, ext = mel_image(seg, sr, n_fft, hop, n_mels, fmax)

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2),
                           gridspec_kw={"width_ratios": [1.9, 1]})
    im = ax[0].imshow(D, origin="lower", aspect="auto", extent=ext, cmap="magma")
    ax[0].set_xlabel("Time (s)"); ax[0].set_ylabel("Mel frequency (Hz)")
    ax[0].set_title(f"{LABELS[k]} cycle  |  {D.shape[0]} mels x {D.shape[1]} frames")
    fig.colorbar(im, ax=ax[0], label="Power (dB re max)")

    sq = np.array(np.clip((D - D.min()) / max(float(np.ptp(D)), 1e-9), 0, 1) * 255, dtype=np.uint8)
    ax[1].imshow(sq, origin="lower", aspect="equal", cmap="gray",
                 extent=[0, 1, 0, 1], interpolation="bilinear")
    ax[1].set_title("as the vision model sees it\n(grey, resized)")
    ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].grid(False)
    plt.show()
    print(f"mel array {D.shape}  (n_mels x frames)   |   on-disk images for this project are 251 x 128")
    print(f"time resolution {hop/sr*1000:.1f} ms/frame   |   window {n_fft/sr*1000:.1f} ms")

In [ ]:
# --- interactive ---
_m_rec  = W.Dropdown(options=list(RECS), value=list(RECS)[0], description="Recording:",
                     layout=W.Layout(width="420px"))
_m_cyc  = W.IntSlider(value=0, min=0, max=1, step=1, description="cycle #:", continuous_update=False)
_m_nfft = W.SelectionSlider(options=[256, 512, 1024, 2048], value=1024, description="n_fft:",
                            continuous_update=False)
_m_hop  = W.SelectionSlider(options=[64, 128, 256, 512], value=256, description="hop:",
                            continuous_update=False)
_m_mels = W.SelectionSlider(options=[32, 64, 128], value=128, description="n_mels:",
                            continuous_update=False)
_m_fmax = W.FloatSlider(value=1900, min=500, max=2000, step=100, description="fmax (Hz):",
                        continuous_update=False, style={"description_width":"110px"})
_out5   = W.Output()

def _refresh5(*_):
    with _out5:
        clear_output(wait=True)
        show_mel(_m_rec.value, _m_cyc.value, 4.0, _m_nfft.value, _m_hop.value,
                 _m_mels.value, _m_fmax.value, SR_D)

def _on_mrec(*_):
    _m_cyc.max = max(len(load_cycles(_m_rec.value)) - 1, 0)
    _m_cyc.value = 0
    _refresh5()

_m_rec.observe(_on_mrec, "value")
for _w in (_m_cyc, _m_nfft, _m_hop, _m_mels, _m_fmax):
    _w.observe(_refresh5, "value")
_on_mrec()
display(W.VBox([W.HBox([_m_rec, _m_cyc]), W.HBox([_m_nfft, _m_hop]),
                W.HBox([_m_mels, _m_fmax]), _out5]))

### ✏️ Try it yourself

Set `n_fft=256` and then `n_fft=2048` on the same crackle cycle. Which one would *you* classify more confidently, and what did you give up to get it? Now set `fmax=1000` — how much of the image is doing no work?

*(The cell below is yours — nothing here is graded.)*

In [ ]:
# Your turn. The helpers above are all available, e.g. load_recording, apply_filter, cut_cycle.

---
## 6 | Train a classifier

**What this does.** Loads one backbone's **pre-computed embeddings**, then trains a small classifier head
on top: 20 epochs, keeping whichever epoch scored best on validation.

**Nothing heavy runs here.** No backbone, no images, no audio, no GPU. Each backbone was run over the
dataset once in advance and its output vectors saved, so switching model in the dropdown just loads a
different array. Training a 2-layer head on a few thousand vectors takes about a second on a CPU.

**What you can try.** Compare a 128-dimensional embedding (`vggish`) against a 1024-dimensional one
(`yamnet`) — bigger is not automatically better. Turn **class weights** on: the dataset is dominated by
normal cycles, and weighting trades overall accuracy for abnormal sensitivity. Push the learning rate to
0.1 and watch it become unstable.

**Two caveats you should know before reading any ranking.**

- `resnet50` and `efficientnet_v2_s` are marked **ImageNet logits** in the dropdown. Their saved vectors
  are 1000-dimensional ImageNet *class scores*, not pooled features, because the head-removal step in the
  original extraction did not take effect for those two architectures. They are kept so you can see the
  cost of tapping the wrong layer — but a ranking that includes them is partly measuring that mistake.
- `opera_ct` has no saved validation split, so the notebook holds out 15% of its training rows with a
  fixed seed. Its validation set is therefore *not* the same cycles the other five models use.
- **The validation split is a random 15% of the training split, so it shares patients with training.**
  The official ICBHI test split *is* patient-disjoint; validation is not. Expect validation scores around
  0.58–0.72 against test scores around 0.45–0.50 — that gap is patient leakage, not overfitting in the
  usual sense. Validation here is good enough to choose an epoch and useless as an estimate of real
  performance. Only the test number means anything.

In [ ]:
# --- embeddings -------------------------------------------------------------
def model_choices():
    """Dropdown entries: (label, key). Labels state what the vectors actually are."""
    out = []
    for key, m in MANIFEST["models"].items():
        d = m["embedding_dim"]
        logits = "logit" in (m.get("caveat") or "").lower() and d == 1000
        tag = f"{d}-d ImageNet LOGITS - not embeddings" if logits else f"{d}-d embeddings"
        out.append((f"{key}  ({tag})", key))
    return sorted(out, key=lambda kv: kv[1])

def load_split(model_key):
    """Return dict with Xtr,ytr,Xva,yva,Xte,yte as float32/int64, plus a note."""
    E = lambda s: np.load(os.path.join(DATA_DIR, "embeddings", f"{model_key}_{s}.npy")).astype(np.float32)
    L = lambda f: np.load(os.path.join(DATA_DIR, f)).astype(np.int64)
    meta = MANIFEST["models"][model_key]
    Xtr  = E("train")
    ytr  = L(meta["train_labels_file"])
    Xte, yte = E("test"), L("labels_test.npy")
    note = ""
    if meta["rows"].get("val"):
        Xva, yva = E("val"), L("labels_val.npy")
    else:
        # OPERA: no saved val. Stratified 15% holdout, fixed seed.
        rng = np.random.default_rng(20260101)
        idx_va = []
        for c in range(len(LABELS)):
            ic = np.where(ytr == c)[0]
            rng.shuffle(ic)
            idx_va.append(ic[:max(1, int(round(0.15 * len(ic))))])
        idx_va = np.sort(np.concatenate(idx_va))
        mask = np.ones(len(ytr), bool); mask[idx_va] = False
        Xva, yva = Xtr[idx_va], ytr[idx_va]
        Xtr, ytr = Xtr[mask],   ytr[mask]
        note = (f"no saved validation split - held out {len(yva)} rows "
                f"(15%, seed 20260101); not the same cycles the other models validate on")
    assert len(Xtr) == len(ytr) and len(Xva) == len(yva) and len(Xte) == len(yte)
    return dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte, note=note)

In [ ]:
# --- a small MLP head, in numpy ---------------------------------------------
# Deliberately dependency-free: no torch, no GPU. The whole point of using
# pre-computed embeddings is that this part is tiny.

EPOCHS, BATCH, HIDDEN = 20, 128, 128

def _onehot(y, k):
    o = np.zeros((len(y), k), np.float32); o[np.arange(len(y)), y] = 1.0; return o

def _softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

def icbhi_scores(y_true, y_pred):
    """Official ICBHI 2017 definition.
    Sp = recall on Normal. Se = pooled recall over all abnormal cycles. Score = mean.
    """
    norm = y_true == NORMAL
    abn  = ~norm
    sp = float((y_pred[norm] == y_true[norm]).mean()) if norm.any() else float("nan")
    se = float((y_pred[abn]  == y_true[abn]).mean())  if abn.any()  else float("nan")
    return se, sp, (se + sp) / 2.0

def train_head(data, optimizer="adam", lr=1e-3, class_weights=False,
               epochs=EPOCHS, hidden=HIDDEN, seed=0, log=print):
    rng = np.random.default_rng(seed)
    Xtr, ytr, Xva, yva = data["Xtr"], data["ytr"], data["Xva"], data["yva"]
    K, d = len(LABELS), Xtr.shape[1]

    mu = Xtr.mean(0); sd = Xtr.std(0); sd[sd < 1e-6] = 1.0        # standardisation from TRAIN only
    Ztr, Zva = (Xtr - mu) / sd, (Xva - mu) / sd

    W1 = (rng.standard_normal((d, hidden)) * np.sqrt(2.0 / d)).astype(np.float32)
    b1 = np.zeros(hidden, np.float32)
    W2 = (rng.standard_normal((hidden, K)) * np.sqrt(2.0 / hidden)).astype(np.float32)
    b2 = np.zeros(K, np.float32)
    P  = [W1, b1, W2, b2]
    state = [[np.zeros_like(p) for p in P], [np.zeros_like(p) for p in P]]

    if class_weights:
        cnt = np.bincount(ytr, minlength=K).astype(np.float32)
        cw = (len(ytr) / (K * np.maximum(cnt, 1))).astype(np.float32)
    else:
        cw = np.ones(K, np.float32)

    def fwd(Z):
        H = np.maximum(Z @ P[0] + P[1], 0.0)
        return H, _softmax(H @ P[2] + P[3])

    best = {"score": -1.0}
    hist = []
    for ep in range(1, epochs + 1):
        order = rng.permutation(len(Ztr))
        for s in range(0, len(order), BATCH):
            ix = order[s:s + BATCH]
            Z, y = Ztr[ix], ytr[ix]
            H, Pr = fwd(Z)
            w = cw[y][:, None]
            G  = (Pr - _onehot(y, K)) * w / len(ix)
            gW2 = H.T @ G;  gb2 = G.sum(0)
            dH  = (G @ P[2].T) * (H > 0)
            gW1 = Z.T @ dH; gb1 = dH.sum(0)
            grads = [gW1, gb1, gW2, gb2]
            for i, g in enumerate(grads):
                if optimizer == "sgd":
                    state[0][i] = 0.9 * state[0][i] + g
                    P[i] -= lr * state[0][i]
                else:  # adam
                    state[0][i] = 0.9  * state[0][i] + 0.1  * g
                    state[1][i] = 0.999 * state[1][i] + 0.001 * (g * g)
                    mh = state[0][i] / (1 - 0.9 ** ep)
                    vh = state[1][i] / (1 - 0.999 ** ep)
                    P[i] -= lr * mh / (np.sqrt(vh) + 1e-8)
        pv = fwd(Zva)[1].argmax(1)
        se, sp, sc = icbhi_scores(yva, pv)
        hist.append((ep, se, sp, sc))
        if sc > best["score"]:
            best = {"score": sc, "epoch": ep, "P": [p.copy() for p in P]}
    log(f"best epoch {best['epoch']}/{epochs}  |  val ICBHI score {best['score']:.4f}")
    return {"params": best["P"], "mu": mu, "sd": sd, "hidden": hidden,
            "best_epoch": best["epoch"], "val_score": best["score"], "history": hist,
            "optimizer": optimizer, "lr": lr, "class_weights": bool(class_weights)}

def predict(model, X):
    W1, b1, W2, b2 = model["params"]
    Z = (X - model["mu"]) / model["sd"]
    H = np.maximum(Z @ W1 + b1, 0.0)
    return _softmax(H @ W2 + b2)

In [ ]:
# --- interactive ---
TRAINED = {}      # filled by the Train button; sections 7 and 8 read it

_t_model = W.Dropdown(options=model_choices(), description="Model:",
                      layout=W.Layout(width="560px"), style={"description_width":"70px"})
_t_opt   = W.Dropdown(options=[("Adam", "adam"), ("SGD + momentum", "sgd")],
                      value="adam", description="Optimizer:")
_t_lr    = W.SelectionSlider(options=[1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1],
                             value=1e-3, description="learning rate:",
                             readout_format=".4f", style={"description_width":"110px"})
_t_cw    = W.Checkbox(value=False, description="class weights (boost rare classes)", indent=False)
_t_btn   = W.Button(description="Train", button_style="primary", icon="play")
_out6    = W.Output()

def _do_train(_=None):
    with _out6:
        clear_output(wait=True)
        key = _t_model.value
        print(f"loading {key} embeddings ...")
        data = load_split(key)
        if data["note"]:
            print(f"  note: {data['note']}")
        print(f"  train {data['Xtr'].shape}  val {data['Xva'].shape}  test {data['Xte'].shape}")
        print(f"training {EPOCHS} epochs, {_t_opt.value}, lr={_t_lr.value:g}, "
              f"class_weights={_t_cw.value} ...")
        m = train_head(data, _t_opt.value, _t_lr.value, _t_cw.value)
        pte = predict(m, data["Xte"]).argmax(1)
        se, sp, sc = icbhi_scores(data["yte"], pte)
        m.update(model_key=key, data=data, test_pred=pte,
                 test_se=se, test_sp=sp, test_score=sc)
        TRAINED.clear(); TRAINED.update(m)
        print(f"\nTEST  Se {se:.4f}   Sp {sp:.4f}   ICBHI score {sc:.4f}")
        ep, s_se, s_sp, s_sc = zip(*m["history"])
        fig, ax = plt.subplots(figsize=(6.4, 2.4))
        ax.plot(ep, s_sc, "-o", ms=3, color="#0b6e75", label="val ICBHI score")
        ax.plot(ep, s_se, lw=0.9, color="#c2622a", label="val Se")
        ax.plot(ep, s_sp, lw=0.9, color="#9aa5ad", label="val Sp")
        ax.axvline(m["best_epoch"], ls="--", lw=0.9, color="#2b3036")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Score"); ax.set_ylim(0, 1)
        ax.set_title(f"{key} — best epoch {m['best_epoch']}")
        ax.legend(fontsize=8); plt.show()
        print("Now run section 7 for the full metrics, or section 8 to export.")

_t_btn.on_click(_do_train)
display(W.VBox([_t_model, W.HBox([_t_opt, _t_lr]), _t_cw, _t_btn, _out6]))

### ✏️ Try it yourself

Train `vggish` (128-d) and then `yamnet` (1024-d) with identical settings. Does the eight-times bigger embedding win? Then retrain the better one **with class weights on** and compare Se against Sp. Which would a clinician rather have?

*(The cell below is yours — nothing here is graded.)*

In [ ]:
# Your turn. The helpers above are all available, e.g. load_recording, apply_filter, cut_cycle.

---
## 7 | Results

**What this does.** Scores the head you just trained on the held-out test set, showing whichever metrics
you tick, plus a sample of individual predictions.

**How to read these.** This dataset is heavily imbalanced — most cycles are normal — so **accuracy is the
least informative number here**. A model that answers "Normal" every time already scores about **57%
accuracy**.

**And the uncomfortable part.** That same do-nothing model scores `Se = 0.00`, `Sp = 1.00`, and therefore
an **ICBHI score of exactly 0.5000** — which is *higher* than any head you will train in this notebook
(they land around 0.45–0.50). That is not a bug in the notebook and not a failure of the embeddings; it is
a property of a metric that averages two rates without regard to class sizes. Hold on to it when you read
any ICBHI leaderboard: a number near 0.5 may mean a model that learned something useful but imperfectly,
or a model that learned nothing at all. The per-class recalls below are what tell those two apart.

- **Sp** (specificity) — recall on normal cycles.
- **Se** (sensitivity) — pooled recall over all abnormal cycles.
- **ICBHI score** — the mean of the two. The headline number for this dataset, and the one to compare.
- **Macro F1** — treats all four classes as equally important, so rare classes count fully.
- **Per-class recall** — where the failures actually are. *Both* is the hardest; there are only 143 of
  them in the test set.

In [ ]:
def confusion(y_true, y_pred, k):
    C = np.zeros((k, k), int)
    for t, p in zip(y_true, y_pred):
        C[t, p] += 1
    return C

def macro_f1(C):
    f1 = []
    for i in range(len(C)):
        tp = C[i, i]; fp = C[:, i].sum() - tp; fn = C[i, :].sum() - tp
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return float(np.mean(f1)), f1

def show_results(which, n_examples=12):
    if not TRAINED:
        print("Nothing trained yet — press Train in section 6 first."); return
    m = TRAINED; d = m["data"]
    yte, pte = d["yte"], m["test_pred"]
    C = confusion(yte, pte, len(LABELS))
    se, sp, sc = m["test_se"], m["test_sp"], m["test_score"]
    acc = float((pte == yte).mean())
    mf1, per_f1 = macro_f1(C)

    print(f"model: {m['model_key']}   |   {m['optimizer']}, lr={m['lr']:g}, "
          f"class_weights={m['class_weights']}   |   best epoch {m['best_epoch']}/{EPOCHS}")
    print(f"test set: {len(yte)} cycles\n")
    rows = []
    if "Se" in which:            rows.append(("Sensitivity (Se)", f"{se:.4f}"))
    if "Sp" in which:            rows.append(("Specificity (Sp)", f"{sp:.4f}"))
    if "ICBHI score" in which:   rows.append(("ICBHI score", f"{sc:.4f}"))
    if "Accuracy" in which:      rows.append(("Accuracy", f"{acc:.4f}"))
    if "Macro F1" in which:      rows.append(("Macro F1", f"{mf1:.4f}"))
    for k, v in rows:
        print(f"  {k:22s} {v}")

    if "Per-class recall" in which:
        print("\n  per-class recall (and F1):")
        for i, n in enumerate(LABELS):
            tot = C[i].sum()
            rc = C[i, i] / tot if tot else float("nan")
            print(f"    {n:9s} n={tot:5d}   recall {rc:.3f}   F1 {per_f1[i]:.3f}")

    if "Confusion matrix" in which:
        fig, ax = plt.subplots(figsize=(4.4, 3.8))
        Cn = C / np.maximum(C.sum(1, keepdims=True), 1)
        im = ax.imshow(Cn, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(LABELS)), LABELS, rotation=45, ha="right")
        ax.set_yticks(range(len(LABELS)), LABELS)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title("Confusion matrix (row-normalised)")
        for i in range(len(LABELS)):
            for j in range(len(LABELS)):
                ax.text(j, i, f"{C[i,j]}", ha="center", va="center", fontsize=8,
                        color="white" if Cn[i, j] > 0.5 else "#2b3036")
        ax.grid(False); fig.colorbar(im, ax=ax, label="fraction of true class")
        plt.show()

    if "Example predictions" in which:
        rng = np.random.default_rng(0)
        ix = rng.choice(len(yte), size=min(n_examples, len(yte)), replace=False)
        pr = predict(m, d["Xte"][ix])
        print(f"\n  {'#':>6s}  {'true':9s} {'predicted':9s} {'conf':>6s}   ok")
        for j, i in enumerate(ix):
            t, p = LABELS[yte[i]], LABELS[pte[i]]
            print(f"  {i:6d}  {t:9s} {p:9s} {pr[j].max():6.2f}   "
                  f"{'yes' if yte[i]==pte[i] else 'NO'}")

In [ ]:
# --- interactive ---
_ALL_METRICS = ["Se", "Sp", "ICBHI score", "Accuracy", "Macro F1",
                "Per-class recall", "Confusion matrix", "Example predictions"]
_r_sel = W.SelectMultiple(options=_ALL_METRICS,
                          value=("Se", "Sp", "ICBHI score", "Per-class recall", "Confusion matrix"),
                          rows=8, description="Show:", layout=W.Layout(width="340px"))
_r_btn = W.Button(description="Show results", icon="table")
_out7  = W.Output()

def _show7(_=None):
    with _out7:
        clear_output(wait=True)
        show_results(set(_r_sel.value))

_r_btn.on_click(_show7)
_r_sel.observe(_show7, "value")
display(W.VBox([W.HBox([_r_sel, _r_btn]), _out7]))
_show7()

---
## 8 | Export to ONNX

**What this does.** Writes the trained head — and only the head — as an ONNX file, plus a small JSON
describing it. In Colab the two files download automatically.

**Standardisation is folded in.** The head was trained on standardised inputs, `(x − μ) / σ`. Rather than
ask whoever loads the model to remember μ and σ, they are absorbed into the first layer's weights:

```
(x − μ)/σ | W₁ + b₁   ≡   x | (W₁/σ) + (b₁ − (μ/σ)·W₁)
```

Same arithmetic, one fewer thing to get wrong. The exported graph therefore takes **raw embeddings** and
returns class probabilities.

**What it is not.** This is the head only — roughly 150 KB. It expects an embedding from the matching
backbone, so the JSON records which backbone, what dimension, and the class order. Without that, the
file is unusable.

In [ ]:
def export_onnx(out_dir=".", filename=None):
    if not TRAINED:
        print("Nothing trained yet — press Train in section 6 first."); return None
    if not ensure("onnx"):
        print("onnx not available; cannot export."); return None
    import onnx
    from onnx import helper, TensorProto, numpy_helper

    m = TRAINED
    W1, b1, W2, b2 = m["params"]
    mu, sd = m["mu"], m["sd"]
    # fold standardisation into layer 1
    W1f = (W1 / sd[:, None]).astype(np.float32)
    b1f = (b1 - (mu / sd) @ W1).astype(np.float32)
    d, K = W1f.shape[0], W2.shape[1]

    init = [numpy_helper.from_array(W1f, "W1"), numpy_helper.from_array(b1f, "b1"),
            numpy_helper.from_array(W2.astype(np.float32), "W2"),
            numpy_helper.from_array(b2.astype(np.float32), "b2")]
    nodes = [helper.make_node("Gemm", ["embedding", "W1", "b1"], ["h_pre"]),
             helper.make_node("Relu", ["h_pre"], ["h"]),
             helper.make_node("Gemm", ["h", "W2", "b2"], ["logits"]),
             helper.make_node("Softmax", ["logits"], ["probabilities"], axis=1)]
    graph = helper.make_graph(
        nodes, f"bsn_head_{m['model_key']}",
        [helper.make_tensor_value_info("embedding", TensorProto.FLOAT, ["N", d])],
        [helper.make_tensor_value_info("probabilities", TensorProto.FLOAT, ["N", K])],
        initializer=init)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)],
                              producer_name="bsn-workshop")
    model.ir_version = 9
    onnx.checker.check_model(model)

    stem = filename or f"bsn_head_{m['model_key']}"
    onnx_path = os.path.join(out_dir, f"{stem}.onnx")
    json_path = os.path.join(out_dir, f"{stem}.json")
    onnx.save(model, onnx_path)

    side = {
        "backbone": m["model_key"],
        "backbone_detail": MANIFEST["models"][m["model_key"]]["architecture"],
        "embedding_dim": int(d),
        "class_order": LABELS,
        "input": {"name": "embedding", "shape": ["N", int(d)], "dtype": "float32",
                  "note": "raw embedding; standardisation is folded into the graph"},
        "output": {"name": "probabilities", "shape": ["N", int(K)], "dtype": "float32"},
        "head": {"hidden_units": int(m["hidden"]), "activation": "relu",
                 "epochs": EPOCHS, "best_epoch": int(m["best_epoch"]),
                 "optimizer": m["optimizer"], "learning_rate": float(m["lr"]),
                 "class_weights": m["class_weights"]},
        "scores": {"val_icbhi_score": float(m["val_score"]),
                   "test_se": float(m["test_se"]), "test_sp": float(m["test_sp"]),
                   "test_icbhi_score": float(m["test_score"])},
        "caveat": MANIFEST["models"][m["model_key"]].get("caveat", ""),
        "dataset": "ICBHI 2017 (Rocha et al., 2019)",
    }
    json.dump(side, open(json_path, "w", encoding="utf-8"), indent=2)

    print(f"{onnx_path}   {os.path.getsize(onnx_path)/1024:.0f} KB")
    print(f"{json_path}   {os.path.getsize(json_path)/1024:.1f} KB")

    if ensure("onnxruntime"):
        import onnxruntime as ort
        s = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        X = m["data"]["Xte"][:64].astype(np.float32)
        got = s.run(None, {"embedding": X})[0]
        ref = predict(m, X)
        print(f"onnxruntime check: max|onnx - numpy| = {np.abs(got - ref).max():.2e}")
    else:
        print("onnxruntime unavailable — graph written but not numerically verified.")

    if IN_COLAB:
        from google.colab import files
        files.download(onnx_path); files.download(json_path)
    return onnx_path, json_path

In [ ]:
# --- interactive ---
_e_btn = W.Button(description="Export ONNX", button_style="success", icon="download")
_out8  = W.Output()

def _do_export(_=None):
    with _out8:
        clear_output(wait=True)
        export_onnx(".")

_e_btn.on_click(_do_export)
display(W.VBox([_e_btn, _out8]))

---
## 9 | Wrap-up

**Five things worth taking home**

1. **Every preprocessing choice is a modelling choice.** The filter band, the segment length, the padding
   mode — you changed each one by hand in sections 3 to 5 and saw the signal change. None of them is a
   neutral default, and all of them have to be written down for a result to mean anything.
2. **Frozen embeddings make this accessible.** Nothing in section 6 needed a GPU, because the expensive
   part ran once, in advance. For many clinical problems, a small head on a frozen backbone is a strong
   baseline and a fast experiment.
3. **The tap point matters as much as the backbone.** Two of the six "embeddings" here are ImageNet class
   scores, because one line of head-removal code silently did nothing. The dimension — exactly 1000 —
   was the only visible clue.
4. **Know what your metric rewards.** Always "Normal" scores ~57% accuracy on this test set — and an
   ICBHI score of exactly 0.5000, which beats every head trained here. Se, Sp and their mean are what the
   challenge reports, but a single averaged rate can be gamed by doing nothing. Report per-class recall
   alongside it, always.
5. **A model is not a file, it is a file plus its contract.** The exported ONNX is useless without the
   backbone name, the embedding dimension and the class order — which is why the JSON ships beside it.

**Questions to argue about**

- The four recording devices sound obviously different. How much of any model's score here is recognising
  the *microphone* rather than the *lung*? How would you design a test that separates the two?
- Reflect padding duplicates real audio to fill a fixed window. Does that leak information a model can
  exploit? Would zeros be more honest, or just differently wrong?
- *Both* has 143 test cycles against 1579 normal. Is a four-class formulation even the right framing, or
  should crackle and wheeze be two independent binary problems — which is how the annotation is actually
  stored?
- You selected the best of 20 epochs using the validation set. What does that do to the validation score
  as an estimate of real performance, and why is the test set reported separately?
- If you deployed the exported head in a clinic, what would you need that this notebook never touched?

---

**Dataset citation.** Rocha, B. M. et al. (2019). *An open access database for the evaluation of
respiratory sound classification algorithms.* Physiological Measurement, 40(3), 035001.

**Known data caveats** are recorded in `manifest.json` and `FINDINGS.md` in the data bundle. Read them
before quoting any number from this notebook.